# Lab 18: The grounded answering service

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Project-Lantern-Case-Study-Lab2-Team-2 | LLM mode: replay


In [2]:
import inspect
print(inspect.getsource(schemas.Number)); print(inspect.getsource(schemas.Claim))
LARGE = llm.P["llm"]["models"]["large"]
FACTS = sd.seller_facts()
blocks = {r["block_id"]: r for r in sd.records("deal_a") if r["doc_id"] == "DR-deal_a-001"}   # block ids are unique only within a document
extracted = {b: llm.structured(schemas.MgmtClaims, prompts.claims_messages(blocks[b]["text"]), LARGE) for b in sd.CLAIM_BLOCKS}
CLAIMS = sd.claim_rows(extracted)

class Number(BaseModel):
    value: float; unit: str
    fact_id: Optional[str] = None    # tiers 1-3
    calc_id: Optional[str] = None    # derived
    claim_id: Optional[str] = None   # tiers 4-5
    @model_validator(mode="after")
    def one_reference(self):
        refs = (self.fact_id, self.calc_id, self.claim_id)
        if sum(r is not None for r in refs) != 1:
            raise ValueError("exactly one reference")
        return self

class Claim(BaseModel):
    text: str
    numbers: list[Number]
    citations: list[Citation]
    tier_label: Literal["filed", "seller", "management claim", "web"]
    @model_validator(mode="after")
    def claim_ids_need_a_label(self):
        ok = ("management claim", "web")
        uses = any(n.claim_id for n in self.numbers)
        if uses and self.tier_label not in ok:
            raise ValueError("claim_id needs a claim label")
        return self



In [3]:
# From the slide: The numeric-claim linter
import math, re
NUM = re.compile(r"""(?P<neg>\()?\$?\s*
  (?P<num>\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)
  \s*(?P<unit>%|x\b|bn\b|mm\b|m\b|million\b|k\b)?
  (?(neg)\))""", re.X | re.I)
YEAR = re.compile(r"^(19|20)\d{2}$")
SCALE = {"bn": 1e9, "mm": 1e6, "m": 1e6, "million": 1e6,
         "k": 1e3}

def numbers_in(text):
    for m in NUM.finditer(text):
        raw, unit = m["num"], (m["unit"] or "").lower()
        a, b = m.start("num"), m.end("num")
        if (YEAR.match(raw) and not unit
                or text[a-1:a].isalpha()   # Q2, FY24
                or re.match(r"-[A-Za-z]", text[b:b+2])
                or re.search(r"(item|note)\s*$",
                             text[max(0, a-6):a], re.I)):
            continue               # 10-K, 8-K, Item 7
        v = float(raw.replace(",", ""))
        v *= SCALE.get(unit, 1)
        yield -v if m["neg"] else v

def lint(text, allowed, tol=0.005):     # [] == pass
    ok = lambda v: any(math.isclose(v, a, rel_tol=tol)
                       for a in allowed)
    return [v for v in numbers_in(text) if not ok(v)]

In [4]:
fact_by_id = {f["fact_id"]: f for f in FACTS}; claim_by_id = {c["claim_id"]: c for c in CLAIMS}
def allowed_for(claim):
    vals = []
    for n in claim.numbers:
        if n.fact_id: f = fact_by_id[n.fact_id]; v, pct = f["value"], f["unit"] == "pct"
        elif n.claim_id: c = claim_by_id[n.claim_id]; v, pct = c["value_abs"], c["unit"] == "pct"
        else: continue                          # calc_id: add your calculator outputs here (Labs 19-20)
        vals += [v, v * 100] if pct else [v]    # the linter reads "24.6%" as 24.6
    return vals
def answer(task_id):
    t = sd.dev_tasks()[task_id]; q = t["turns"][0]["content"]
    msgs = prompts.answer_messages(q, FACTS, CLAIMS, sd.evidence_for(task_id))
    ans = llm.structured(schemas.Answer, msgs, LARGE)
    bad = [v for c in ans.claims for v in lint(c.text, allowed_for(c))]
    if bad:                                         # regenerate once with the linter's feedback
        print(f"  linter rejected {bad}; regenerating")
        ans = llm.structured(schemas.Answer, prompts.regenerate_messages(msgs, ans.model_dump_json(), bad), LARGE)
        keep = [c for c in ans.claims if not lint(c.text, allowed_for(c))]
        if len(keep) < len(ans.claims): ans = schemas.Answer(claims=keep, abstained=not keep, abstain_reason="partially abstained")
    return q, ans
Path("outputs/qa").mkdir(parents=True, exist_ok=True)
with open("outputs/qa/dev.jsonl", "w") as fh:
    for tid in sd.TASK_EVIDENCE:
        q, ans = answer(tid); print(f"Q: {q}")
        print("  Not disclosed." if ans.abstained and not ans.claims else "", ans.abstain_reason or "")
        for c in ans.claims: print(f"  [{c.tier_label}] {c.text}")
        fh.write(json.dumps({"task_id": tid, "answer": ans.model_dump()}) + "\n")

Q: What was Wren's FY2025 revenue?
 
  [seller] Wren's FY2025 revenue was $41.2 million, per seller financials.
  linter rejected [118.0]; regenerating
Q: What was Wren's net revenue retention in FY2025, per the seller's KPIs?
 
  [seller] Net revenue retention was 109% in FY2025, per seller financials.
  [management claim] Per the CIM (unverified), net revenue retention was 118%, which management calls its best quarter annualized.
Q: What is Wren's FY2026 budgeted EBITDA?
  Not disclosed. Not disclosed: the data room does not include an FY2026 budget.


The service from the slide, written to `services/qa/app.py`. Run it with `uvicorn services.qa.app:app` from the repository root.

In [5]:
service = """import sys, json
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from src.schemas import Answer
from src.llm.client import MODE
from src import sample_data as sd
facts = {f["fact_id"]: f for f in sd.seller_facts()}
def grounded_answer(question, scope):
    return Answer(claims=[], abstained=True, abstain_reason="Not disclosed: wire in the notebook-18 pipeline here.")
""" + 'from fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel\n\napp = FastAPI(title="LANTERN QA")\nALLOWED_SCOPES = {"deal_a", "deal_b", "kestrel", "public"}\n\nclass Ask(BaseModel):\n    question: str\n    scope: str                   # required, no default\n\n@app.post("/answer", response_model=Answer)\ndef answer(req: Ask) -> Answer:\n    if req.scope not in ALLOWED_SCOPES:\n        raise HTTPException(400, "unknown scope")\n    # grounded_answer: retrieve, generate, lint, cite\n    return grounded_answer(req.question, req.scope)\n@app.get("/facts/{fact_id}")        # and /calc/{calc_id}\ndef fact(fact_id: str):\n    if fact_id not in facts:\n        raise HTTPException(404, "unknown fact")\n    return facts[fact_id]\n@app.get("/health")\ndef health():\n    return {"ok": True, "llm_mode": MODE}'
Path("services/qa").mkdir(parents=True, exist_ok=True); Path("services/qa/app.py").write_text(service)
try:
    from fastapi.testclient import TestClient
    import importlib.util
    spec = importlib.util.spec_from_file_location("qa_app", "services/qa/app.py"); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    cl = TestClient(m.app)
    print(cl.get("/health").json(), cl.post("/answer", json={"question": "q", "scope": "nowhere"}).status_code)
except ImportError:
    print("pip install fastapi httpx to test the service here")

{'ok': True, 'llm_mode': 'replay'} 400


**Exercise 18 — measure extraction separately from evidence checking.**
First retain the ten existing tests of `lint(text, allowed)`: they check whether extracted values are supported. Then test `numbers_in(text)` directly with five strings whose numbers must be extracted, five that must be skipped, and three additional traps (date, page reference, count). A correctly extracted number can still be rejected when it lacks evidence.

In [8]:
import pandas as pd

# Each allowed list represents evidence supplied to the linter.
tests = [
    # Five cases that should pass
    ("Revenue was $41.2 million in 2025.", [41_200_000], True),
    ("The 10-K reports revenue of $41.2 million.", [41_200_000], True),
    ("FY2025 growth was 24.6%.", [24.6], True),
    ("The adjustment was ($1.6 million).", [-1_600_000], True),
    ("The company serves 1,240 customers.", [1240], True),

    # Five cases that should fail
    ("Revenue was $50 million.", [41_200_000], False),
    ("Growth was 32%.", [24.6], False),
    ("The adjustment was ($2 million).", [-1_600_000], False),
    ("The company serves 1,500 customers.", [1240], False),
    ("The multiple was 4x.", [3], False),
]

results = []

for text, allowed, expected_pass in tests:
    rejected = lint(text, allowed)
    actual_pass = len(rejected) == 0

    results.append({
        "text": text,
        "numbers_detected": list(numbers_in(text)),
        "rejected_numbers": rejected,
        "expected": "accept" if expected_pass else "reject",
        "actual": "accept" if actual_pass else "reject",
        "correct": actual_pass == expected_pass,
    })

results_df = pd.DataFrame(results)
display(results_df)

print(f"Tests passed: {results_df['correct'].sum()}/{len(tests)}")

,text,numbers_detected,rejected_numbers,expected,actual,correct
0,Revenue was $41.2 million in 2025.,[41200000.0],[],accept,accept,True
1,The 10-K reports revenue of $41.2 million.,[41200000.0],[],accept,accept,True
2,FY2025 growth was 24.6%.,[24.6],[],accept,accept,True
3,The adjustment was ($1.6 million).,[-1600000.0],[],accept,accept,True
4,"The company serves 1,240 customers.",[1240.0],[],accept,accept,True
5,Revenue was $50 million.,[50000000.0],[50000000.0],reject,reject,True
6,Growth was 32%.,[32.0],[32.0],reject,reject,True
7,The adjustment was ($2 million).,[-2000000.0],[-2000000.0],reject,reject,True
8,"The company serves 1,500 customers.",[1500.0],[1500.0],reject,reject,True
9,The multiple was 4x.,[4.0],[4.0],reject,reject,True


Tests passed: 10/10


In [9]:
# Hand-labelled expected extractions, chosen before running the harness.
cases = [
    ("extract", "$52.3 million", [52_300_000.0]),
    ("extract", "($1,200)", [-1200.0]),
    ("extract", "14.2x", [14.2]),
    ("extract", "24.6%", [24.6]),
    ("extract", "$1.2bn", [1_200_000_000.0]),
    ("skip", "FY2025", []),
    ("skip", "Item 7", []),
    ("skip", "Q2", []),
    ("skip", "10-K", []),
    ("skip", "in 2019", []),
    ("date trap", "March 31, 2026", []),
    ("page trap", "on page 41", []),
    ("count trap", "The company serves 1,240 customers.", [1240.0]),
]
extraction_rows = []
for kind, text, expected in cases:
    detected = list(numbers_in(text))
    extraction_rows.append({
        "case": kind, "text": text, "expected": expected,
        "detected": detected, "correct": detected == expected,
    })
extraction_results = pd.DataFrame(extraction_rows)
display(extraction_results)
base = extraction_results.iloc[:10]
traps = extraction_results.iloc[10:]
assert base["correct"].all(), "One of the ten basic extraction/skip cases regressed."
print(f"Basic extraction/skip cases: {base['correct'].sum()}/10")
print(f"Additional traps: {traps['correct'].sum()}/3")
print(f"All extraction cases: {extraction_results['correct'].sum()}/{len(cases)}")
print("Failures exposed by this exercise:")
display(extraction_results.loc[~extraction_results["correct"], ["case", "text", "expected", "detected"]])


,case,text,expected,detected,correct
0,extract,$52.3 million,[52300000.0],[52300000.0],True
1,extract,"($1,200)",[-1200.0],[-1200.0],True
2,extract,14.2x,[14.2],[14.2],True
3,extract,24.6%,[24.6],[24.6],True
4,extract,$1.2bn,[1200000000.0],[1200000000.0],True
5,skip,FY2025,[],[],True
6,skip,Item 7,[],[],True
7,skip,Q2,[],[],True
8,skip,10-K,[],[],True
9,skip,in 2019,[],[],True


Basic extraction/skip cases: 10/10
Additional traps: 1/3
All extraction cases: 11/13
Failures exposed by this exercise:


,case,text,expected,detected
10,date trap,"March 31, 2026",[],[31.0]
11,page trap,on page 41,[],[41.0]


In [10]:
# Counts and derived multiples require supporting evidence, even when parsed correctly.
count_text = "The company serves 1,240 customers."
assert list(numbers_in(count_text)) == [1240.0]
assert lint(count_text, [1240.0]) == []
assert lint(count_text, []) == [1240.0]

statement = "grew 4× since founding in 2019"
print("Derived-growth statement extracts:", list(numbers_in(statement)))
print("Without supporting evidence, rejected values:", lint(statement, []))
assert list(numbers_in(statement)) == [4.0]
assert lint(statement, []) == [4.0]


Derived-growth statement extracts: [4.0]
Without supporting evidence, rejected values: [4.0]


**Findings.** The five extraction cases and five skip cases all pass. Of the three extra traps, the customer count is handled correctly, but the day `31` and page number `41` are incorrectly treated as factual numeric claims. The result is **11/13 exact-match extraction cases**, with two false alarms. The existing **10/10 evidence-checking tests** do not detect these extraction errors because they do not cover date and page references. This small exercise is not a substitute for CS2's precision measurement on at least 50 labelled numbers.

A page number or day-of-month should not trigger a financial fact lookup. Here both can cause needless regeneration or dropped claims. The classroom regex is retained so the exercise documents these limitations; improving it requires context-aware date/page handling and additional tests. Counts such as 1,240 customers should still be extracted and supported, not blanket-skipped.

**“Grew 4× since founding in 2019”: bug or correct refusal?** The extractor skips the year but finds the growth multiple. Rejecting 4× is correct when no cited fact or validated calculation supports it. If starting and ending values support the ratio, compute it, cite its inputs, and make its `calc_id` resolvable by the linter. Notebook 18's current `allowed_for` skips `calc_id`, so supporting that derived claim requires calculator integration. The model should not guess the missing evidence.
